In [ ]:
# Import library here
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [11]:
df = pd.read_csv("../../data/processed/01.3/accidents_advance_clean.csv")
df['Start_Time'] = pd.to_datetime(df['Start_Time'])
print(f"Shape: {df.shape}")
df.head(5)

Shape: (5469092, 42)


,Severity,Start_Time,Start_Lat,Start_Lng,Distance(mi),Description,Street,City,County,State,...,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight,Duration(min),Start_Date,Hour,Month,Weather_Group
0,3,2016-11-08 08:50:37,29.718655,-95.321053,0.009950,Accident on I-45 Northbound at I-45 Exits 43A ...,I-45 N,Houston,Harris,TX,...,False,Day,Day,Day,Day,3.405079,2016-11-08,8,11,Cloudy
1,2,2021-07-26 07:04:56,38.927551,-121.080093,0.000000,Right hand shoulder blocked due to accident on...,Taylor Ln,Auburn,Placer,CA,...,False,Day,Day,Day,Day,3.862833,2021-07-26,7,7,Clear
2,2,2018-10-24 08:23:38,34.776241,-86.672829,0.000000,Lane blocked due to accident on AL-255 Researc...,Highway 255,Huntsville,Madison,AL,...,False,Day,Day,Day,Day,3.422633,2018-10-24,8,10,Clear
3,3,2019-07-11 10:42:48,42.384880,-83.149467,0.470004,Entry ramp to I-96 Westbound from Davison West...,Edward J Jeffries Fwy,Detroit,Wayne,MI,...,False,Day,Day,Day,Day,4.784571,2019-07-11,10,7,Clear
4,2,2020-11-06 01:29:00,33.776575,-117.837134,0.387980,1023: NB 55 JNO 22. SV REAR ENDED VV AND WAS L...,Garden Grove Fwy,Orange,Orange,CA,...,False,Night,Night,Night,Night,4.952535,2020-11-06,1,11,Clear


In [12]:
#Duration bins
df['Duration_bin'] = pd.cut(
    df['Duration(min)'],
    bins=[0, 15, 60, 180, np.inf],
    labels=['Short(<15)', 'Medium(15-60)', 'Long(60-180)', 'Very_Long(>180)']
)

print("Target variable created:")
print(df[['Duration(min)', 'Distance(mi)', 'Duration_bin']].head(10))

Target variable created:
   Duration(min)  Distance(mi) Duration_bin
0       3.405079      0.009950   Short(<15)
1       3.862833      0.000000   Short(<15)
2       3.422633      0.000000   Short(<15)
3       4.784571      0.470004   Short(<15)
4       4.952535      0.387980   Short(<15)
5       5.758376      0.033435   Short(<15)
6       4.665638      0.000000   Short(<15)
7       4.248495      1.015955   Short(<15)
8       5.332880      0.000000   Short(<15)
9       3.751463      0.000000   Short(<15)


สร้าง target ใหม่ ใช้กับ regression เพราะ scale ข้อมูลให้สมมาตรขึ้น ทำให้ model อย่าง Linear Regression หรือ XGBoost เรียนรู้ได้ดีกว่า และ Duration_bin ใช้ถ้าอยากเปลี่ยนเป็น classification แทน เช่น ทำนายแค่ว่า "สั้น / กลาง / ยาว / นานมาก"


In [ ]:
# Temporal Features
df['Hour']       = df['Start_Time'].dt.hour
df['Month']      = df['Start_Time'].dt.month
df['DayOfWeek']  = df['Start_Time'].dt.dayofweek   # 0=Mon
df['Year']       = df['Start_Time'].dt.year
df['Quarter']    = df['Start_Time'].dt.quarter

# Binary convenience features
df['IsWeekend']  = df['DayOfWeek'].isin([5, 6]).astype(int)
df['IsRushHour'] = df['Hour'].isin([7, 8, 9, 16, 17, 18]).astype(int)
df['IsNight']    = (df['Sunrise_Sunset'] == 'Night').astype(int)

# Cyclical encoding (keeps hour 23 close to hour 0)
df['Hour_sin']   = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_cos']   = np.cos(2 * np.pi * df['Hour'] / 24)
df['Month_sin']  = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos']  = np.cos(2 * np.pi * df['Month'] / 12)
df['DOW_sin']    = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['DOW_cos']    = np.cos(2 * np.pi * df['DayOfWeek'] / 7)

print("Temporal features:")
print(df[['Hour','IsRushHour','IsWeekend','IsNight',
          'Hour_sin','Hour_cos','Month_sin','Month_cos']].head(5))

Temporal features:
   Hour  IsRushHour  IsWeekend  IsNight  Hour_sin  Hour_cos  Month_sin  \
0     8           1          0        0  0.866025 -0.500000  -0.500000   
1     7           1          0        0  0.965926 -0.258819  -0.500000   
2     8           1          0        0  0.866025 -0.500000  -0.866025   
3    10           0          0        0  0.500000 -0.866025  -0.500000   
4     1           0          0        1  0.258819  0.965926  -0.500000   

   Month_cos  
0   0.866025  
1  -0.866025  
2   0.500000  
3  -0.866025  
4   0.866025  


สร้าง 3 กลุ่ม คือ binary flags (IsRushHour, IsWeekend, IsNight) ที่บอกว่าตรงเงื่อนไขไหม, cyclical encoding (sin/cos) ที่แก้ปัญหาว่า "ชั่วโมง 23 ใกล้กับชั่วโมง 0 มากกว่าชั่วโมง 12" แต่ถ้าใส่ตัวเลขตรงๆ model จะเข้าใจผิด และ raw values (Hour, Month, DayOfWeek) สำหรับ tree-based models ที่ไม่ต้องการ cyclical


In [14]:
#Weather Features
# Weather group encoding
weather_map = {
    'Clear': 0, 'Cloudy': 1, 'Rain': 2,
    'Snow_Ice': 3, 'Fog': 4, 'Severe': 5,
    'Windy': 6, 'Other': 7
}
df['Weather_encoded'] = df['Weather_Group'].map(weather_map).fillna(7)

# Risk level from weather (domain knowledge)
weather_risk = {
    'Clear': 1, 'Cloudy': 2, 'Windy': 2,
    'Rain': 3, 'Fog': 4, 'Snow_Ice': 5,
    'Severe': 5, 'Other': 2
}
df['Weather_Risk'] = df['Weather_Group'].map(weather_risk).fillna(2)

# Interaction: bad weather + night
df['Weather_Night_Risk'] = df['Weather_Risk'] * (df['IsNight'] + 1)

# Visibility bucket
df['Visibility_bucket'] = pd.cut(
    df['Visibility(mi)'],
    bins=[-1, 0.25, 1, 5, 10, np.inf],
    labels=[0, 1, 2, 3, 4]
).astype(int)

# Temperature bucket (Fahrenheit)
df['Temp_bucket'] = pd.cut(
    df['Temperature(F)'],
    bins=[-np.inf, 32, 50, 70, 90, np.inf],
    labels=['freezing', 'cold', 'mild', 'warm', 'hot']
)

print("Weather features:")
print(df[['Weather_Group','Weather_encoded','Weather_Risk',
          'Weather_Night_Risk','Visibility_bucket']].head(8))

Weather features:
  Weather_Group  Weather_encoded  Weather_Risk  Weather_Night_Risk  \
0        Cloudy                1             2                   2   
1         Clear                0             1                   1   
2         Clear                0             1                   1   
3         Clear                0             1                   1   
4         Clear                0             1                   2   
5        Cloudy                1             2                   2   
6        Cloudy                1             2                   2   
7         Clear                0             1                   2   

   Visibility_bucket  
0                  3  
1                  4  
2                  4  
3                  4  
4                  4  
5                  4  
6                  4  
7                  4  


แปลง weather จาก text เป็นตัวเลขที่มีความหมาย แทนที่จะ encode แบบ 0/1/2/3 ธรรมดา (ซึ่ง model ไม่รู้ว่า Fog อันตรายกว่า Rain) ใส่ domain knowledge เข้าไปว่า Fog/Snow อันตรายกว่า Clear มาก และ Weather_Night_Risk เป็น interaction term  อากาศแย่ + กลางคืน = อันตรายมากกว่าอย่างใดอย่างหนึ่งแบบ additive

In [15]:
#Road & Infrastructure Features
road_features = ['Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction',
                 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop',
                 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop']

# Convert bool → int
for col in road_features:
    if col in df.columns:
        df[col] = df[col].astype(int)

# Road complexity score — sum of all features at the location
df['Road_Complexity'] = df[road_features].sum(axis=1)

# High-risk road combo (junction + no exit = trapped)
df['Is_Trapped']      = ((df['Junction'] == 1) & (df['No_Exit'] == 1)).astype(int)

# Controlled intersection (has traffic signal or stop)
df['Is_Controlled']   = ((df['Traffic_Signal'] == 1) | (df['Stop'] == 1)).astype(int)

print("Road complexity distribution:")
print(df['Road_Complexity'].value_counts().sort_index())
print(f"\nTrapped intersections: {df['Is_Trapped'].sum():,}")

Road complexity distribution:
Road_Complexity
0    3795928
1    1112131
2     450799
3      92249
4      15659
5       2172
6        153
7          1
Name: count, dtype: int64

Trapped intersections: 727


Road_Complexity นับว่าจุดนั้นมีสิ่งก่อสร้างทางถนนกี่อย่าง ยิ่งเยอะยิ่งซับซ้อน และ Is_Trapped คือ domain insight ที่สำคัญมาก  Junction คือทางแยก, No_Exit คือไม่มีทางออก ถ้าเกิดอุบัติเหตุตรงนี้รถจะติดค้าง ช่วยเหลือยาก duration จะนานเป็นพิเศษ

In [16]:
#Geographic Features
# State encoding (frequency encoding)
state_freq = df['State'].value_counts(normalize=True)
df['State_freq'] = df['State'].map(state_freq)

# City encoding (frequency encoding — avoids high-cardinality problem)
city_freq = df['City'].value_counts(normalize=True)
df['City_freq'] = df['City'].map(city_freq)

# Lat/Lng grid (500m bins) — chronic hotspot location feature
GRID = 0.005
df['lat_grid'] = (df['Start_Lat'] / GRID).round(0) * GRID
df['lng_grid'] = (df['Start_Lng'] / GRID).round(0) * GRID
df['grid_id']  = df['lat_grid'].astype(str) + "_" + df['lng_grid'].astype(str)

# Chronic hotspot flag — cells with ≥5 accidents + ≥2 years Condition
grid_stats = df.groupby('grid_id').agg(
    n_accidents=('Severity', 'count'),
    n_years=('Year', 'nunique'),
    avg_severity=('Severity', 'mean')
).reset_index()

chronic_grids = set(grid_stats[
    (grid_stats['n_accidents'] >= 5) &
    (grid_stats['n_years'] >= 2) &
    (grid_stats['avg_severity'] >= 2.5)
]['grid_id'])

df['Is_Chronic_Hotspot'] = df['grid_id'].isin(chronic_grids).astype(int)

# Accident frequency at this grid cell
grid_freq = df['grid_id'].value_counts()
df['Grid_Accident_Freq'] = df['grid_id'].map(grid_freq)

print(f"Chronic hotspot grids: {len(chronic_grids):,}")
print(f"Accidents at chronic spots: {df['Is_Chronic_Hotspot'].sum():,}")
print(f"  = {df['Is_Chronic_Hotspot'].mean()*100:.1f}% of all accidents")

Chronic hotspot grids: 21,275
Accidents at chronic spots: 1,037,312
  = 19.0% of all accidents


แปลง location จาก lat/lng ที่มีค่าเยอะมาก ให้เป็น grid cell 500m ก่อน จากนั้น flag ว่าจุดนั้นเป็น chronic hotspot ไหม ส่วน State_freq และ City_freq คือ frequency encoding แทน one-hot encoding เพราะ State มี 50 ค่า City มีเป็นพัน ถ้า one-hot จะได้ column เพิ่มหลายพัน column ซึ่งทำให้ model ช้าและ overfit ง่าย

In [17]:
#Severity & Interaction Features
# Severity as risk proxy (already numeric 1-4)
df['High_Severity'] = (df['Severity'] >= 3).astype(int)

# Key interactions for predicting Duration
df['Rush_x_Severity']     = df['IsRushHour'] * df['Severity']
df['Night_x_Severity']    = df['IsNight'] * df['Severity']
df['Weather_x_Severity']  = df['Weather_Risk'] * df['Severity']
df['Complexity_x_Severity'] = df['Road_Complexity'] * df['Severity']
df['Hotspot_x_Severity']  = df['Is_Chronic_Hotspot'] * df['Severity']

print("Interaction features created:")
print(df[['Rush_x_Severity','Night_x_Severity',
          'Weather_x_Severity','Hotspot_x_Severity']].describe().round(2))

Interaction features created:
       Rush_x_Severity  Night_x_Severity  Weather_x_Severity  \
count       5469092.00        5469092.00          5469092.00   
mean              0.91              0.68                3.98   
std               1.14              1.07                2.33   
min               0.00              0.00                1.00   
25%               0.00              0.00                2.00   
50%               0.00              0.00                4.00   
75%               2.00              2.00                4.00   
max               4.00              4.00               20.00   

       Hotspot_x_Severity  
count          5469092.00  
mean                 0.51  
std                  1.08  
min                  0.00  
25%                  0.00  
50%                  0.00  
75%                  0.00  
max                  4.00  


สร้าง interaction term โดยคูณ feature สองตัวเข้าหากัน เหตุผลคือ rush hour + severity 4 ไม่ใช่แค่ "บวกกัน" แต่ "คูณกัน"  อุบัติเหตุรุนแรงในช่วง rush hour แย่กว่าอุบัติเหตุเดียวกันตอนตีสองมาก Linear model จะ capture ไม่ได้ถ้าไม่สร้าง term นี้ให้ก่อน

In [19]:
TARGET_DURATION = 'Duration(min)' 
TARGET_DISTANCE = 'Distance(mi)'

# รายชื่อ Feature ทั้งหมดที่เตรียมไว้จากขั้นตอน Feature Engineering
FEATURES = [
    'Hour', 'Month', 'DayOfWeek', 'Quarter', 'Year',
    'IsWeekend', 'IsRushHour', 'IsNight',
    'Hour_sin', 'Hour_cos', 'Month_sin', 'Month_cos',
    
    'Severity', 'High_Severity',
    
    'Weather_encoded', 'Weather_Risk', 'Weather_Night_Risk',
    'Visibility_bucket', 'Humidity(%)', 'Precipitation(in)',
    
    'Road_Complexity', 'Is_Trapped', 'Is_Controlled',
    'Junction', 'Traffic_Signal', 'Crossing', 'No_Exit',
    
    'State_freq', 'City_freq', 'Is_Chronic_Hotspot', 'Grid_Accident_Freq',
    
    'Rush_x_Severity', 'Night_x_Severity',
    'Weather_x_Severity', 'Hotspot_x_Severity',
]

FEATURES = [f for f in FEATURES if f in df.columns]

X = df[FEATURES].copy()
y_dur = df[TARGET_DURATION].copy() 
y_dis = df[TARGET_DISTANCE].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Features used ({len(FEATURES)}):")
print(FEATURES)
print(f"\nMissing values in X: {X.isnull().sum().sum()}")

Feature matrix shape: (5469092, 35)
Features used (35):
['Hour', 'Month', 'DayOfWeek', 'Quarter', 'Year', 'IsWeekend', 'IsRushHour', 'IsNight', 'Hour_sin', 'Hour_cos', 'Month_sin', 'Month_cos', 'Severity', 'High_Severity', 'Weather_encoded', 'Weather_Risk', 'Weather_Night_Risk', 'Visibility_bucket', 'Humidity(%)', 'Precipitation(in)', 'Road_Complexity', 'Is_Trapped', 'Is_Controlled', 'Junction', 'Traffic_Signal', 'Crossing', 'No_Exit', 'State_freq', 'City_freq', 'Is_Chronic_Hotspot', 'Grid_Accident_Freq', 'Rush_x_Severity', 'Night_x_Severity', 'Weather_x_Severity', 'Hotspot_x_Severity']

Missing values in X: 0
